# Demo: `forecast()` method

This notebook demonstrates the `ForecastingAssistant.forecast()` method for every supported forecaster type.

The `forecast()` method:
1. Profiles the data
2. Generates a plan
3. Generates code
4. Executes the code via `exec()`
5. Returns a `RunResult` with predictions, metrics, and the exact code that was run

**Key guarantee:** `result.code` is the exact script that produced `result.predictions` and `result.metrics`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")
assistant = ForecastingAssistant()

## Synthetic Data

We create datasets for each forecaster type.

In [3]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series with exogenous variable
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)
exog = rng.normal(50, 10, n)

df_single = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": exog,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

# Wide format for multivariate
df_wide = pd.DataFrame({
    "date": dates[:n_multi],
    "temperature": 20 + np.cumsum(rng.normal(0, 0.5, n_multi)),
    "humidity": 60 + np.cumsum(rng.normal(0, 0.3, n_multi)),
    "pressure": 1013 + np.cumsum(rng.normal(0, 0.2, n_multi)),
})

print(f"df_single: {df_single.shape}")
print(f"df_multi:  {df_multi.shape}")
print(f"df_wide:   {df_wide.shape}")

df_single: (365, 3)
df_multi:  (600, 3)
df_wide:   (200, 4)


---
## 1. ForecasterRecursive (single series)

The default for single-series forecasting. Uses recursive multi-step prediction.

In [4]:
result_recursive = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14
)

print("Plan:", result_recursive.plan.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions:")
display(result_recursive.predictions)

Plan: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 2, 3, 4, 5, 7, 8]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (raw ordinal encoding). NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,2.931103,10.108784,1.010196,0.03041



Predictions:


,pred
2022-12-18,98.425777
2022-12-19,98.256442
2022-12-20,94.580389
2022-12-21,91.675341
2022-12-22,89.362358
2022-12-23,91.150429
2022-12-24,95.323472
2022-12-25,98.371999
2022-12-26,97.448323
2022-12-27,94.428085


In [5]:
result_recursive = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14,
    forecaster  = "ForecasterRecursive",
    estimator   = "LGBMRegressor",
)

print("Plan:", result_recursive.plan.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions:")
display(result_recursive.predictions)

Plan: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 2, 3, 4, 5, 7, 8]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (raw ordinal encoding). NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,2.931103,10.108784,1.010196,0.03041



Predictions:


,pred
2022-12-18,98.425777
2022-12-19,98.256442
2022-12-20,94.580389
2022-12-21,91.675341
2022-12-22,89.362358
2022-12-23,91.150429
2022-12-24,95.323472
2022-12-25,98.371999
2022-12-26,97.448323
2022-12-27,94.428085


In [6]:
# Inspect the generated code (this is exactly what was executed)
print(result_recursive.code)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures, CalendarFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2022-12-17'  # last training date, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['promo']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean

In [7]:
# Inspect the generated plan
print(result_recursive.plan)

task_type='single_series' forecaster='ForecasterRecursive' forecaster_kwargs={'lags': [1, 2, 3, 4, 5, 7, 8], 'window_features': [{'stats': ['mean', 'std'], 'window_sizes': 3}, {'stats': ['mean'], 'window_sizes': 7}, {'stats': ['mean'], 'window_sizes': 21}], 'calendar_features': {'features': ['day_of_week', 'weekend', 'month'], 'encoding': None}, 'categorical_features': 'auto', 'dropna_from_series': False} estimator='LGBMRegressor' estimator_kwargs={} steps=14 frequency='D' end_train='2022-12-17' interval=None interval_method=None metric='mean_absolute_error' metrics_to_compute=['mean_absolute_error', 'mean_squared_error', 'mean_absolute_scaled_error', 'mean_absolute_percentage_error'] use_exog=True preprocessing_steps=[] warnings=[] explanation="Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 2, 3, 4, 5, 7, 8]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (raw ordinal encoding). Na

---
## 2. ForecasterDirect (single series)

Trains one model per step. Better for long horizons or when the DGP changes across horizons.

In [8]:
result_direct = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14,
    forecaster  = "ForecasterDirect",
    estimator   = "Ridge",
)

print("Plan:", result_direct.plan.explanation)
print("\nMetrics:")
display(result_direct.metrics)
print("\nPredictions:")
display(result_direct.predictions)

Plan: Plan: ForecasterDirect + Ridge. Lags: [1, 2, 3, 4, 5, 7, 8]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (cyclical encoding). NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,0.877651,1.188531,0.30248,0.009218



Predictions:


,pred
2022-12-18,99.479348
2022-12-19,100.624850
2022-12-20,97.747603
2022-12-21,93.311892
2022-12-22,90.499325
2022-12-23,92.043970
2022-12-24,95.672271
2022-12-25,100.209373
2022-12-26,101.287987
2022-12-27,98.543286


---
## 3. ForecasterRecursiveMultiSeries (multi-series)

Global model trained on all series simultaneously. One row per level in metrics.

In [9]:
result_multi = assistant.forecast(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    steps            = 7,
    test_size        = 7,
    forecaster       = "ForecasterRecursiveMultiSeries",
    estimator        = "Ridge",
)

print("Plan:", result_multi.plan.explanation)
print("\nMetrics (per series):")
display(result_multi.metrics)
print("\nPredictions:")
display(result_multi.predictions)

Plan: Plan: ForecasterRecursiveMultiSeries + Ridge. Lags: [1, 2, 3, 4, 5, 7]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (cyclical encoding). NaN rows kept (NaN-tolerant estimator). MASE is scale-independent, enabling fair comparison across differently-scaled series.

Metrics (per series):


,series,MAE,MSE,MASE,MAPE
0,store_A,0.733304,0.733635,0.934730,0.007465
1,store_B,1.873361,6.454637,1.694492,0.008442
2,store_C,1.603915,2.944562,2.355613,0.011834



Predictions:


,level,pred
2022-07-13,store_A,98.202171
2022-07-13,store_B,220.592724
2022-07-13,store_C,136.772081
2022-07-14,store_A,98.601836
2022-07-14,store_B,220.164927
2022-07-14,store_C,136.967688
2022-07-15,store_A,99.043415
2022-07-15,store_B,219.642184
2022-07-15,store_C,137.127423
2022-07-16,store_A,99.377393


---
## 4. ForecasterDirectMultiVariate (multivariate)

Uses all series as features to predict one target series. Direct strategy.

In [10]:
result_multivariate = assistant.forecast(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
    steps       = 7,
    test_size   = 7,
    forecaster  = "ForecasterDirectMultiVariate",
    estimator   = "Ridge",
)

print("Plan:", result_multivariate.plan.explanation)
print("\nMetrics (per series):")
display(result_multivariate.metrics)
print("\nPredictions:")
display(result_multivariate.predictions)

Plan: Plan: ForecasterDirectMultiVariate + Ridge. Lags: [1, 2, 3, 4, 5, 7]. Window features: ['mean(window=3)', 'std(window=3)', 'mean(window=7)', 'mean(window=21)']. Calendar features: ['day_of_week', 'weekend', 'month'] (cyclical encoding). NaN rows kept (NaN-tolerant estimator). MASE is scale-independent, enabling fair comparison across differently-scaled series.

Metrics (per series):


,series,MAE,MSE,MASE,MAPE
0,temperature,0.589655,0.463115,1.394022,0.027852



Predictions:


,level,pred
2022-07-13,temperature,21.721670
2022-07-14,temperature,22.008906
2022-07-15,temperature,22.197157
2022-07-16,temperature,22.223268
2022-07-17,temperature,22.159617
2022-07-18,temperature,21.926558
2022-07-19,temperature,21.700887


In [11]:
print(result_multivariate.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures, CalendarFeatures
from skforecast.direct import ForecasterDirectMultiVariate

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2022-07-12'  # last training date, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'mean'],
    window_sizes = [3, 3, 7, 21],
)

calendar_features = CalendarFeatures(
    features = ['day_of_week', 'weekend', 'month'],
    encoding = 'cyclical',
)

# Create f

---
## 5. ForecasterStats (ARIMA)

Statistical model. No external estimator needed. Supports native prediction intervals.

In [12]:
result_stats = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14,
    forecaster  = "ForecasterStats",
    interval    = [0.1, 0.9],
)

print("Plan:", result_stats.plan.explanation)
print("\nMetrics:")
display(result_stats.metrics)
print("\nPredictions with intervals:")
display(result_stats.predictions)

Plan: Plan: ForecasterStats + Arima. Prediction intervals via native. Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,2.198992,5.596321,0.757876,0.022886



Predictions with intervals:


,pred,lower_bound,upper_bound
2022-12-18,100.093746,97.654693,102.532798
2022-12-19,98.849512,94.959952,102.739072
2022-12-20,96.118451,91.089435,101.147467
2022-12-21,92.488220,86.611207,98.365234
2022-12-22,88.850461,82.169529,95.531392
2022-12-23,90.859215,83.514661,98.203768
2022-12-24,94.891653,86.892864,102.890443
2022-12-25,99.513353,89.937671,109.089036
2022-12-26,98.542013,87.294797,109.789228
2022-12-27,95.961653,83.238260,108.685046


---
## 6. ForecasterFoundation (zero-shot)

Pre-trained foundation model (Chronos-2). No training — uses the series context to forecast directly.
Requires `pip install chronos-forecasting` and downloads model weights on first use.

In [13]:
result_foundation = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14,
    forecaster  = "ForecasterFoundation",
    interval    = [0.1, 0.9],
)

print("Plan:", result_foundation.plan.explanation)
print("\nMetrics:")
display(result_foundation.metrics)
print("\nPredictions with quantile intervals:")
display(result_foundation.predictions)

Plan: Plan: ForecasterFoundation + Chronos-2. Prediction intervals via native. Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,3.254085,11.892719,1.121511,0.033958



Predictions with quantile intervals:


,level,q_0.1,q_0.5,q_0.9
2022-12-18,sales,96.712837,98.050453,99.194672
2022-12-19,sales,97.283531,99.022072,100.599724
2022-12-20,sales,94.175560,95.981033,97.750694
2022-12-21,sales,88.412537,90.944786,93.132027
2022-12-22,sales,84.973541,88.093140,90.881699
2022-12-23,sales,85.803543,89.046616,91.830276
2022-12-24,sales,90.339920,93.478577,96.286713
2022-12-25,sales,94.731995,98.025162,100.987770
2022-12-26,sales,95.266708,98.743195,101.800499
2022-12-27,sales,92.227310,95.676865,98.793053


---
## 7. ForecasterRecursive with prediction intervals

ML forecaster with bootstrapping-based prediction intervals.

In [14]:
result_intervals = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    test_size   = 14,
    forecaster  = "ForecasterRecursive",
    estimator   = "Ridge",
    interval    = [0.1, 0.9],
)

print("Interval method:", result_intervals.plan.interval_method)
print("\nMetrics:")
display(result_intervals.metrics)
print("\nPredictions with 80% intervals:")
display(result_intervals.predictions)

Interval method: bootstrapping

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,0.714364,0.917885,0.246203,0.007508



Predictions with 80% intervals:


,pred,lower_bound,upper_bound
2022-12-18,99.339785,98.544009,100.192300
2022-12-19,100.335099,98.787795,101.688786
2022-12-20,97.689687,95.654566,99.190321
2022-12-21,93.392699,91.375301,95.256862
2022-12-22,90.596004,88.125229,92.335190
2022-12-23,91.834409,89.417092,93.798906
2022-12-24,96.019785,93.592649,97.951672
2022-12-25,100.269480,97.655353,102.298217
2022-12-26,101.469724,98.682296,103.879665
2022-12-27,98.821417,96.006347,101.042252


---
## Comparing metrics across forecasters

Since all `RunResult.metrics` share the same schema, we can easily compare.

In [15]:
comparison = pd.concat([
    result_recursive.metrics.assign(forecaster="ForecasterRecursive"),
    result_direct.metrics.assign(forecaster="ForecasterDirect"),
    result_stats.metrics.assign(forecaster="ForecasterStats"),
    result_foundation.metrics.assign(forecaster="ForecasterFoundation"),
    result_intervals.metrics.assign(forecaster="ForecasterRecursive + intervals"),
], ignore_index=True)

display(comparison[["forecaster", "series", "MAE", "MSE", "MASE"]])

,forecaster,series,MAE,MSE,MASE
0,ForecasterRecursive,sales,2.931103,10.108784,1.010196
1,ForecasterDirect,sales,0.877651,1.188531,0.302480
2,ForecasterStats,sales,2.198992,5.596321,0.757876
3,ForecasterFoundation,sales,3.254085,11.892719,1.121511
4,ForecasterRecursive + intervals,sales,0.714364,0.917885,0.246203


---
## Inspecting the generated code

Every `RunResult` includes the full standalone script. You can copy-paste it and run independently.

In [16]:
# Show code for the multi-series forecaster
print(result_multi.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures, CalendarFeatures, reshape_series_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

# Train/test split
end_train = '2022-07-12'  # last training date, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.in